<a href="https://colab.research.google.com/github/rafahcs/Projeto_AF/blob/main/projetoaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#RASCUNHO DE IDEIAS PARA O CÓDIGO

#teste

# Números inteiros: sequência de um ou mais dígitos, opcionalmente com sinal. Exemplos: 123, -456, +789

# Função de Transição -> delta
# O template usa um dicionário com uma tupla na chave associada a um valor. Porém, o trabalho pede que o a especificação do AF seja de uma 5-upla.
#Então acho que o valor deve ser uma tripla. A chave é tupla do conjunto de estados com o alfabeto. A tripla seria da função, estado inicial e
#conjunto dos estados finais. Acho que essa tupla é chave, porque com o conjunto dos estados concluímos quem é o inicial e o(s) final(is) e com o
#alfabeto os retornos das funções.

#No codigo
# delta : Dict[(str, str), (str, str, str)]

# (str, str): é a tupla chave que representa o (estado_atual, simbolo_lido)
# (str, str, str): é a tripla dos valores. O retorno da função de transição  deve ser uma tupla contendo:
#	1. Próximo estado
#	2. Símbolo a ser escrito (sobrescrevendo o atual)
#	3. Direção do movimento (Esquerda 'L' ou Direita 'R')

#Definição do estado inicial e preparação da fita
# _init_(self, tape)
#self é o ponteiro
#initial_state guarda o estado inicial
#tape é a fita(lista)
#current_state guarda o estado atual
#len(tape) estado final
#step(self) Marcador de estados

#leitura: armazenameno do estado
# read_state()
#consulta: veirificar estado atual
# select_state()
#atualizar estado fazendo:
#		-escrever novo símbolo na fita
#		-mover o ponteiro self incrementando
# update_state()

#Extra:
#Simulação de Fita Infinita:
#A fita é uma lista. Para ser infinita acho que podemos adicionar um símbolo vazio, toda vez que tentar ler o tamanho da lista.
#Função para gerar instâncias de testes automáticos:
#Isso me lembra o padrão de projeto template method. Consiste em ter um método template que instância vários métodos chamados em uma sequência. Podemos pegar essa ideia


1. INTRODUÇÃO
− Linguagem escolhida
− Motivação

* Foi escolhida a linguagem dos números ...

2. DEFINIÇÃO FORMAL DO AF
− 5−upla completa
− Diagrama de estados (opcional)

Q: Um conjunto finito de estados.

Σ: Um alfabeto finito de símbolos (alfabeto da fita).

δ: Função de transição

qO: Estado inicial

F: Conjunto de estados finais ou de aceitação

M = {Q, Σ, δ, qO, F}

Q = {q0, q1}

Σ = {0, 1}

δ = {}

q0 = ?

F = {?}


3. IMPLEMENTAÇÃO
− Código completo da classe FiniteAutomata
− Funções auxiliares


In [17]:
from typing import Dict, Tuple


class FiniteAutomata:
    delta: Dict[Tuple[str, str], str] = None

#inicialização do automato
    def __init__(self, tape, initial_state, final_states, delta):
        self.tape = list(tape)  #cria fita
        self.blank = "_"
        self.head = 0 #cabeça de leitura da fita

        self.initial_state = initial_state  #estado inicial
        self.current_state = initial_state  #estado atual
        self.final_states = final_states  #estados finais
        self.delta = delta  #estados de aceitação

#leitura de símbolo
    def read_symbol(self):
        if self.head >= len(self.tape):
            return self.blank
        return self.tape[self.head]

#função de transição
    def step(self):
        symbol = self.read_symbol()
        key = (self.current_state, symbol)

        if key not in self.delta:
            return False

        self.current_state = self.delta[key]
        self.head += 1
        return True

    def execute(self):
        while self.head < len(self.tape):
            if not self.step():
                return False
        return self.current_state in self.final_states

# --------------------------------------------------
# INTEIRO
# --------------------------------------------------
def build_integer_af(tape):
    delta = {}
    digits = "0123456789"

    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"

    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    return FiniteAutomata(tape, "q0", {"q2"}, delta)


# --------------------------------------------------
# REAL
# --------------------------------------------------
def build_real_af(tape):
    delta = {}
    digits = "0123456789"

    # início
    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", ".")] = "q3"

    # número inteiro inicial
    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    # ponto depois de número → aceita "3."
    delta[("q2", ".")] = "q5"

    # ponto sem número antes → precisa de dígito depois
    for d in digits:
        delta[("q3", d)] = "q4"

    # parte decimal normal
    for d in digits:
        delta[("q4", d)] = "q4"
        delta[("q5", d)] = "q4"

    # sinal seguido de ponto
    delta[("q1", ".")] = "q3"

    return FiniteAutomata(
        tape,
        "q0",
        {"q2", "q4", "q5"},  # <-- chave da correção
        delta
    )

# --------------------------------------------------
# CIENTÍFICA
# --------------------------------------------------
def build_scientific_af(tape):
    delta = {}
    digits = "0123456789"

    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", ".")] = "q3"

    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"
        delta[("q3", d)] = "q4"
        delta[("q4", d)] = "q4"

    delta[("q1", ".")] = "q3"
    delta[("q2", ".")] = "q3"
    delta[("q3", "e")] = "q5"
    delta[("q3", "E")] = "q5"
    delta[("q2", "e")] = "q5"
    delta[("q2", "E")] = "q5"
    delta[("q4", "e")] = "q5"
    delta[("q4", "E")] = "q5"

    delta[("q5", "+")] = "q6"
    delta[("q5", "-")] = "q6"

    for d in digits:
        delta[("q5", d)] = "q7"
        delta[("q6", d)] = "q7"
        delta[("q7", d)] = "q7"

    return FiniteAutomata(tape, "q0", {"q7"}, delta)

# --------------------------------------------------
# HEXADECIMAL
# --------------------------------------------------
def build_hexadecimal_af(tape):
    delta = {}
    hex_digits = "0123456789abcdefABCDEF"

    # 0x...
    delta[("q0", "0")] = "q1"
    delta[("q1", "x")] = "q2"
    delta[("q1", "X")] = "q2"

    # #ABC
    delta[("q0", "#")] = "q3"

    for d in hex_digits:
        if d != "0": # Se for 0, já definimos que vai para q1
            delta[("q0", d)] = "q4"
        delta[("q1", d)] = "q4" # Começando direto (ex: 1A3)
        delta[("q2", d)] = "q4"
        delta[("q3", d)] = "q4"
        delta[("q4", d)] = "q4"

    delta[("q4", "h")] = "q5"
    delta[("q4", "H")] = "q5"

    return FiniteAutomata(tape, "q0", {"q4", "q5"}, delta)


# --------------------------------------------------
# COMPLEXO
# --------------------------------------------------
def build_complex_af(tape):
    delta = {}
    digits = "0123456789"

    # --- INÍCIO ---
    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", "i")] = "q6"

    # número inicial (parte real OU imaginária direta)
    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    # ponto decimal (opcional)
    delta[("q2", ".")] = "q3"
    for d in digits:
        delta[("q3", d)] = "q3"

    # número seguido de i → forma "bi"
    delta[("q2", "i")] = "q6"
    delta[("q3", "i")] = "q6"

    # parte real seguida de + ou -
    delta[("q2", "+")] = "q4"
    delta[("q2", "-")] = "q4"
    delta[("q3", "+")] = "q4"
    delta[("q3", "-")] = "q4"

    # após sinal da parte imaginária:
    delta[("q4", "i")] = "q6"  # aceita "3+i"

    for d in digits:
        delta[("q4", d)] = "q5"

    # parte imaginária numérica
    for d in digits:
        delta[("q5", d)] = "q5"

    delta[("q5", ".")] = "q4"
    delta[("q5", "i")] = "q6"

    return FiniteAutomata( tape, "q0", {"q6"}, delta )

# construtor dos automatos
automata = {
        "Inteiro": build_integer_af,
        "Real": build_real_af,
        "Cientifico": build_scientific_af,
        "Hexadecimal": build_hexadecimal_af,
        "Complexo": build_complex_af
    }
# --------------------------------------------------
# TESTES
# --------------------------------------------------
def run_tests():
    tests = {
        "Inteiro": [
            ("123", True),
            ("+12", True),
            ("-10", True),
            ("+abc", False) #Não aceita
        ],
        "Real": [
            ("3.14", True),
            ("-0.5", True),
            ("+2.0", True),
            ("5.", True),
            (".5", True),
            ("..5", False)  #Não aceita
        ],
        "Cientifico": [
            ("1.23e4", True),
            ("-5E-10", True),
            ("6E+7", True),
            ("5e3", True),
            ("2.e-2", True),
            ("1e", False), #Não aceita
            ("1e+", False), #Não aceita
            ("e10", False)  #Não aceita
        ],
        "Hexadecimal": [
            ("0xFF", True),
            ("0x1A3", True),
            ("#ABC", True),
            ("1A3h", True),
            ("0x", False) #Não aceita
        ],
        "Complexo": [
            ("2", False), #Não aceita
            ("3+4i", True),
            ("2-10i", True),
            ("3+i", True),
            ("+4i", True),
            ("2.5-1.3i", True)
        ]
    }

    for name, builder in automata.items():
        print(f"\n=== {name} ===")
        for t, expected in tests.get(name, []):
            result = builder(t).execute()

            if result == expected:
                print(f"OK   {t}")
            else:
                print(f"FAIL {t} esperado {expected}, obteve {result}")


'''
Caso de aceitação:
  - Palavras " 0 "(Todos) ou " 0x "(HEXA)
Casos de rejeição:
  - Palavra começando por letra -> rejeitado por todos
'''


# --------------------------------------------------
# EXECUÇÃO
# --------------------------------------------------
if __name__ == "__main__":
    run_tests()


=== Inteiro ===
OK   123
OK   +12
OK   -10
OK   +abc

=== Real ===
OK   3.14
OK   -0.5
OK   +2.0
OK   5.
OK   .5
OK   ..5

=== Cientifico ===
OK   1.23e4
OK   -5E-10
OK   6E+7
OK   5e3
OK   2.e-2
OK   1e
OK   1e+
OK   e10

=== Hexadecimal ===
OK   0xFF
OK   0x1A3
OK   #ABC
OK   1A3h
OK   0x

=== Complexo ===
FAIL 2 esperado True, obteve False
OK   3+4i
OK   2-10i
OK   3+i
OK   +4i
OK   2.5-1.3i


4. TESTES
− Casos pequenos
− Verificação de correção


In [ ]:
automata = {
        "Inteiro": build_integer_af,
        "Real": build_real_af,
        "Científico": build_scientific_af,
        "Hexadecimal": build_hexadecimal_af,
        "Complexo": build_complex_af
    }

def run_tests():
  # test_mal_formada()
  # test_pequenas_instancias()

    print("Teste com pequenas instancias")
# def test_pequenas_instancias():
    tests = [
        "123566", "-456", "+0.456",
        "3.14", ".5",
        "1.2e10",
        "0xFF", "#ABC", "1A3h",
        "3+4i", "2.5-1.3i",
        "abc", "0xG", "3+i"
    ]

    for name, builder in automata.items():
        print(f"\n=== {name} ===")
        for t in tests:
            result = builder(t).execute()
            print(f"{t:12} -> {result}\t" + name)
            if t == tests[len(tests)-1]:
              break

    print("\nTeste de palavras mal formadas\n")
# def test_mal_formada():
    tests = [
        "ABC", ".23", "+-30", "+ABC",
        "-ABC", "+-ABC"
    ]

    for name, builder in automata.items():
      for t in tests:
          result = builder(t).execute()
          print(f"{t} -> {result}\t" + name)
          print("teste falhou")
          if t == "+-ABC":
            break

    print("\nTeste de casos óbvios de aceitação/rejeição")

    tests = [
        "0", "0x", "ABCDE3456"
    ]

    for name, builder in automata.items():
      for t in tests:
          match t :
            case "0":
              result = builder(t).execute()
              print(f"{t} -> {result}\t" + name)

            case "0x":
              result = builder(t).execute()
              print(f"{t} -> {result}\t" + name)

            case _:
              result = builder(t).execute()
              print(f"{t} -> {result}\t" + name)



'\nimport numpy as np\n\nsort = np.arange(0, 10)\nprint(sort)\n'

5. DEMONSTRAÇÃO FORMAL
− Prova de corretude do AF apresentado


6. CONCLUSÃO
− Principais aprendizados
− Limitações do estudo